# Customer walkthrough: V01 through V04

Start with one use case. Each notebook contains its own network diagram, setup examples, executable checks and cleanup steps. Opening a notebook does not change your network or start compute.

| Your requirement | Open | DNS choice | Application traffic |
| --- | --- | --- | --- |
| Read a private Amazon S3 object | [V01_Amazon_S3.ipynb](V01_Amazon_S3.ipynb) | Explicit entry for the actual public S3 hostname | Workspace private connection -> customer subnet -> NAT -> S3 HTTPS |
| Keep an OCI IAM Network Source restriction | [V02_Customer_Network_Source.ipynb](V02_Customer_Network_Source.ipynb) | Explicit entry for the in-region service hostname | Workspace private connection -> customer subnet -> regional service gateway -> OCI service |
| Read Object Storage in another OCI region | [V03_Cross_Region_Object_Storage.ipynb](V03_Cross_Region_Object_Storage.ipynb) | Explicit entry for the remote regional hostname | Workspace private connection -> customer subnet -> NAT -> remote Object Storage HTTPS |
| Use a database private endpoint | [V04_Service_Private_Endpoint.ipynb](V04_Service_Private_Endpoint.ipynb) | Default private DNS proxy for the private-only database name | Workspace private connection -> customer private route -> database listener |

AIDP manages CoreDNS. You choose the private network and, when needed, destination FQDNs in workspace settings. You do not install a CoreDNS server. DNS chooses the destination address; routes carry application traffic, security rules allow connections, and service credentials/IAM authorize the operation.

## Common Workbench actions

### Prepare once

1. **Network administrator:** identify an existing customer VCN/subnet with the required route and security rules. Verify the AIDP private-network prerequisites and the destination controls in the notebook's appendix.
2. **Service owner:** provide a small test object or an actual database private endpoint, plus the existing runtime authorization. A local OCI CLI profile is not the Spark runtime's service identity.
3. **Tester:** open **Workspaces**, select the test workspace, and inspect its private network configuration. Record the VCN, subnet, NSGs, complete Advanced DNS list and SCAN settings. Retain a screenshot or configuration export as the baseline.
4. Open the selected notebook from **Shared**. Use its compute selector to attach the intended test compute. Start it only when the target, permissions and network are ready. Record whether it was already running.
5. Read the numbered steps below. Run **Test helpers** once, fill **Your configuration**, then run that configuration cell once. Keep `RUN_LIVE=False` while preparing. Use the separate **Choose phase** cell for subsequent runs; do not rerun configuration and clear your earlier results.

### Add an explicit destination for V01, V02 or V03

1. In the workspace's edit form, expand **Advanced options** and enable the DNS-name control if necessary. Enter the exact hostname from the client's endpoint, for example `objectstorage.<region>.oraclecloud.com`. Enter no `https://`, path, port, IP address or wildcard.
2. Preserve all existing DNS and SCAN entries. If the exact entry already exists, record that baseline; do not delete an in-use entry to manufacture a failing test.
3. Save the workspace change. Follow any compute-stop requirement shown by the deployment. Inspect the operation result, then reopen the form and confirm the saved entry. A saved row alone is not proof that destination mapping succeeded.
4. Return to the same notebook. If a required restart cleared the session, save your previous results privately and initialize a new session. Otherwise retain `RESULTS`. Set the next phase and rerun **Run a workload phase** with a fresh connection/client as needed after DNS caches expire.

Explicit destinations must resolve from the AIDP service environment during mapping creation. Private-only names that need the customer resolver follow the default private DNS proxy path. Do not add a private database hostname as a troubleshooting shortcut.

### Understand the output

| Output | Customer decision |
| --- | --- |
| `functional: PASS` | The exact fixture rows or expected database identity were returned. |
| `functional: BLOCKED` | Complete the named prerequisite, then repeat this phase. |
| `functional: FAIL` | Inspect the error privately; verify endpoint, route, permission and credential separately. |
| `source_path: BLOCKED` | Supply and review independent path evidence before claiming the intended route. This can coexist with a successful read. |
| `mtls: BLOCKED` in V04 | The query does not prove client-certificate use. Review transport evidence only if making an mTLS claim. |

Keep the time, phase and operation result. A DNS answer or open TCP port is not an authenticated test. A successful functional check can still leave the overall case `INCOMPLETE` pending path/configuration evidence.


[Full tester guide](README.md).
